# 02 · Good ranking, equal decisions?

**Synthetic data.** Kaggle CIBMTR competition data, generated by SurvivalGAN. Results are a proposal to test on real CIBMTR data.

## Question

The competition scored ranking only (stratified C-index). Does good ranking give equally good decisions in each race group?

## Key

| Term | Meaning |
|---|---|
| HCT | Hematopoietic cell transplantation. |
| EFS | Event-free survival. Event = relapse, graft failure or death. |
| C-index | Ranking quality. Fraction of patient pairs in correct risk order. 0.5 = random. |
| Stratified C-index | Competition metric: mean − SD of C-index across race groups. |
| O/E | Observed ÷ expected risk. 1 = correct on average. < 1 = model overpredicts. |
| Calibration slope | 1 = risks spread correctly. < 1 = too extreme. > 1 = too modest. |
| DCA | Decision curve analysis. Net benefit of acting on the model, across thresholds. |
| p_t | Threshold probability. Risk above which you act. |
| Out-of-fold | Each prediction comes from a model that did not see that patient. |

## Method

- Horizon: 12 months. Outcome: EFS event by 12 months.
- Model: 1 Cox model, ridge penalty, no tuning. Same features as competition models, race group included.
- 5-fold out-of-fold predictions on `train.csv` (`test.csv` has no outcomes).
- Censoring: observed risk = Kaplan-Meier (KM) at 12 months. 0.8% censored before 12 months; slope drops them. DCA: time-to-event `dcurves`.

In [ ]:
import warnings
from functools import partial

from hct_equity import eda, fairness, fairness_plots
from hct_equity.config import load_config, load_labels
from hct_equity.data import feature_columns, load_train
from hct_equity.readable import readable
from hct_equity.run_fairness import build_results, score

# Known, handled: rare categories fold into "infrequent"; that column has low variance.
warnings.filterwarnings("ignore", message="Found unknown categories")
warnings.filterwarnings("ignore", message=".*have very low variance")

project = load_config()
cols, settings = project.columns, project.fairness
labels = load_labels()
print(project.synthetic_note)

## 1 · Data and model

Cohort per race group. Groups have equal size by design (synthetic).

In [ ]:
train = load_train(project)
summary = partial(eda.cohort_summary, cols=cols, features=feature_columns(project, train))
display(readable(eda.by_group(train, cols.group, summary), labels))
scored = score(project, train)  # ~40 s: 5 Cox fits
results = build_results(project, scored)

## 2 · C-index by race (competition metric)

Baseline. Same metric the competition scored.

In [ ]:
print(f"Stratified C-index: {results.stratified_c_index:.3f}")
readable(results.metrics, labels)

## 3 · Calibration by race, 12 months

Risk deciles per group: mean predicted vs KM observed. On the diagonal = correct.

In [ ]:
fairness_plots.plot_calibration(results.deciles, cols, settings.plot, project.synthetic_note);

## 4 · DCA by race, 12 months

Act-or-not decision, e.g. closer follow-up after HCT. Act if predicted risk > p_t.
- Raw net benefit tracks each group's event rate. So compare **gain**: model − better of treat all / treat none.
- Gain > 0: model adds value for that group.

In [ ]:
fairness_plots.plot_dca_by_group(results.dca, cols, settings.plot, project.synthetic_note)
readable(fairness.net_benefit_gain(results.dca, cols, settings.table_thresholds), labels, cell_format="signed_three_decimals")

## 5 · Exploratory: per-group recalibration

Does recalibrating each group restore net benefit? Cross-fitted logistic recalibration per group. Exploratory: not part of the main claim.

In [ ]:
recalibrated = scored.assign(**{cols.risk: fairness.recalibrated_risk(scored, cols, settings.horizon_months, settings.folds, settings.seed)})
recalibrated_dca = fairness.dca_by_group(recalibrated, cols, settings.horizon_months, settings.dca_thresholds.values())
display(readable(fairness.group_metrics(recalibrated, cols, settings.horizon_months)[["observed_expected", "calibration_slope"]], labels))
gain = partial(fairness.net_benefit_gain, columns=cols, thresholds=settings.table_thresholds)
readable(gain(recalibrated_dca) - gain(results.dca), labels, cell_format="signed_three_decimals")

## 6 · Result

Numbers from the run of 2026-10-12. 1 model; synthetic data.

- **What differs:** calibration is near-equal (O/E 0.97–1.04, plot near the diagonal). Decision value is not: at p_t 0.6, gain is 0.07 for More than one race vs 0.17 for White.
- **By how much, and why:** White has the lowest C-index (0.648) but the highest gain at p_t 0.6; group order changes with p_t. Per-group recalibration changes gain by ≤ 0.011, so event rate and ranking drive the gap, not calibration.
- **Why it matters:** the stratified C-index cannot show if a model helps each group equally for a given decision. Proposal: repeat on real CIBMTR data; add the 24-month horizon and top competition models.